In [94]:
import mne
import numpy as np
from numpy import pi
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from scipy.signal import welch

# quantum comp sci
import pennylane as qml
from pennylane import numpy as pnp

In [95]:
p1 = mne.io.read_raw('../../data/1/1.edf', preload=True)
a1 = pd.read_csv('../../data/1/1_1.txt', header=None)

Extracting EDF parameters from ../../data/1/1.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 5279999  =      0.000 ... 26399.995 secs...


In [96]:
print(f"Channels p1: {p1.ch_names}") # C4 is in

Channels p1: ['LOC-A2', 'ROC-A1', 'F3-A2', 'C3-A2', 'O1-A2', 'F4-A1', 'C4-A1', 'O2-A1', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'DC3', 'X7', 'X8', 'SaO2', 'DC8']


In [97]:
print(f"Unique annotations: {a1[0].unique()}")

Unique annotations: [0 1 2 3 5]


In [98]:
# check length of 
def print_len_match(raw, annot):
    fs = raw.info["sfreq"]
    print(f"Length: {raw.n_times / fs / len(annot)}")

print_len_match(p1, a1)

Length: 30.0


In [99]:
def load_subject_for_training(edf_path, label_path, channel="C4-A1"):
    raw = mne.io.read_raw(edf_path, preload=True)
    fs = int(raw.info["sfreq"])
    signal = raw.get_data(picks=channel)[0]
    labels = pd.read_csv(label_path, header=None).values.reshape(-1)
    epoch_length = 30

    epochs = signal.reshape(len(labels), fs * epoch_length)
    assert len(signal) == len(labels) * fs * epoch_length
    return epochs, labels, fs

p1, a1, fs1 = load_subject_for_training('../../data/1/1.edf', '../../data/1/1_1.txt')

Extracting EDF parameters from ../../data/1/1.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 5279999  =      0.000 ... 26399.995 secs...


In [100]:
print(p1.shape, np.bincount(a1))

(880, 6000) [264  73 194 231   0 118]


In [101]:
# working with sleep frequency bands
# delta =    0.5 - 4     |   deep sleep (N3). Large, slow waves
# theta =    4-8         |   drowsiness, N1, REM
# alpha =    8-12        |   relaxed wakefulness with eyes closed, fades as we fall asleep
# sigma =    12-16       |   N2 spindle area

BANDS = {
    "delta" : (0.5, 4),     # N3 marker: slow waves of 6s to score block as N3
    "theta" : (4, 8),       # fast waves 
    "alpha" : (8, 12),      # fast waves
    "sigma" : (12, 16),     # N2 marker: slow waves with spindles
}

def band_features(epochs, fs):
    # power spectrum per epoch: 4s windows, 50% overlap, averaged
    freqs, psd = welch(epochs, fs=fs, nperseg=4 * fs, axis = -1) # psd: (n_epochs, 401)
    
    feats = []
    for lo, hi in BANDS.values():
        in_band = (freqs >= lo) & (freqs < hi) # mask over the frequencies
        feats.append(psd[:, in_band].mean(axis=1)) # mean power in band per epoch

    # return log for outliers - one outlier could skew everyting
    return np.log(np.stack(feats, axis=1)) # (n_epochs, 4)

In [102]:
F = band_features(p1, fs1)
print(F.shape)
for i, name in enumerate(BANDS):
    print(f"{name}, N2 : {F[a1 == 2, i].mean():.2f} || N3 : {F[a1 == 3, i].mean():.2f}")

(880, 4)
delta, N2 : -28.40 || N3 : -27.52
theta, N2 : -29.92 || N3 : -29.83
alpha, N2 : -30.68 || N3 : -30.83
sigma, N2 : -31.43 || N3 : -31.53


In [103]:
def prep(edf, txt):
    epochs, labels, fs = load_subject_for_training(edf, txt)
    keep = np.isin(labels, [2,3]) # we want N2 and N3 only for this experiment
    X = band_features(epochs[keep], fs) # keep only N2 adn N3
    y = (labels[keep] == 3).astype(int) # annotate N2 = false and N3 = true
    return X, y

Xa, ya = prep('../../data/1/1.edf', '../../data/1/1_1.txt')
Xb, yb = prep('../../data/2/2.edf', '../../data/2/2_1.txt')
X_test, y_test = prep('../../data/3/3.edf', '../../data/3/3_1.txt')

X_train, y_train = np.vstack([Xa, Xb]), np.concatenate([ya, yb])
print(f"Train: {np.bincount(y_train)}, Test: {np.bincount(y_test)}")

sc = MinMaxScaler(feature_range=(0, np.pi)).fit(X_train) # define scaler map every band between [0, pi] reason is: we are doing rotations (flip our data)
X_train = sc.transform(X_train) # scale the values
X_test = np.clip(sc.transform(X_test), 0, np.pi) # capping below 0 and above pi to keep the rotation in tact

clf = LogisticRegression().fit(X_train, y_train) 
print(f"Logistic Regression balanced accuracy: {balanced_accuracy_score(y_test, clf.predict(X_test))}")

Extracting EDF parameters from ../../data/1/1.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 5279999  =      0.000 ... 26399.995 secs...
Extracting EDF parameters from ../../data/2/2.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 5783999  =      0.000 ... 28919.995 secs...
Extracting EDF parameters from ../../data/3/3.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 5657999  =      0.000 ... 28289.995 secs...
Train: [539 388], Test: [246 173]
Logistic Regression balanced accuracy: 0.8427557685981484


In [104]:
clf_delta = LogisticRegression().fit(X_train[:, [0]], y_train)
print(f"delta only: {balanced_accuracy_score(y_test, clf_delta.predict(X_test[:, [0]]))}")
# delta waves are the most important waves... the other basically just add noise. the task comes down to a 1 variable check

delta only: 0.8592391559753748


In [120]:
n_qubits, n_layers = 4, 2 # we have 4 qubits for our 4 freq. bands, 2 layers (like in a neural network)
dev = qml.device("default.qubit", wires=n_qubits) # exact simulator, torch.device("cpu") quantum edition

@qml.qnode(dev) # -> basically define a torch.nn.Module and the circuit function is the forward function, difference is we store the weights separately
def circuit(weights, x):
    # AngleEmbedding = input layer (similar to an input layer in a neural network)
    # we rotate the qubits
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y") # feature i -> RY(x_i) on qubit i
    # StronglyEntanglingLayers = hidden layer -> we give every qubit 3 trainable rotation angles
    # rotations = per-qubit weights, CNOTs = connections between qubits. 
    # 2*4*3 = 24 trainable weights
    # the CNOTs happen in this strongly entangling layer
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits)) # trainable rotations + CNOTs
    # we measure the qubits
    return qml.expval(qml.PauliZ(0)) # read qubit 0 -> value in [-1, 1]

In [121]:
shape = qml.StronglyEntanglingLayers.shape(n_layers=n_layers, n_wires=n_qubits)  # (2, 4, 3)
print(shape)
print(qml.draw(circuit, level="device")(pnp.zeros(shape), X_train[0]))

(2, 4, 3)
0: ──RY(2.01)──Rot(0.00,0.00,0.00)─╭●───────╭X──Rot(0.00,0.00,0.00)─╭●────╭X────┤  <Z>
1: ──RY(2.73)──Rot(0.00,0.00,0.00)─╰X─╭●────│───Rot(0.00,0.00,0.00)─│──╭●─│──╭X─┤     
2: ──RY(2.94)──Rot(0.00,0.00,0.00)────╰X─╭●─│───Rot(0.00,0.00,0.00)─╰X─│──╰●─│──┤     
3: ──RY(1.78)──Rot(0.00,0.00,0.00)───────╰X─╰●──Rot(0.00,0.00,0.00)────╰X────╰●─┤     


In [122]:
def model(weights, bias, X):
    return circuit(weights, X) + bias # we can pass everything at once - closed form solution style

def loss(weights, bias, X, t):
    return pnp.mean((t - model(weights, bias, X)) ** 2) # use mse quantum edition

t_train = 2 * y_train - 1 # targets: N2 -> -1, N3 -> +1

In [123]:
pnp.random.seed(42)
weights = pnp.random.uniform(0,2 * pnp.pi, size=shape, requires_grad=True)  # we manually i guess we manually define that we need grad on the weights, similar to nn.Parameter
bias = pnp.array(0.0, requires_grad=True) # requires autograd => similar to nn.Parameter again
opt = qml.AdamOptimizer(stepsize=0.05) # just like with torch
rng = np.random.default_rng(42)

def predict(X):
    return (np.asarray(model(weights, bias, X)) > 0).astype(int)

for step in range(100):
    idx = rng.choice(len(X_train), size=64, replace=False) # batch size of 64 with random selection
    (weights, bias), cost = opt.step_and_cost(loss, weights, bias, X=X_train[idx], t=t_train[idx])
    if step % 10 == 0:
        print(f"Step: {step}, Loss {cost:.3f}, Test bal acc {balanced_accuracy_score(y_test, predict(X_test)):.3f}")

Step: 0, Loss 0.901, Test bal acc 0.593
Step: 10, Loss 0.737, Test bal acc 0.685
Step: 20, Loss 0.641, Test bal acc 0.851
Step: 30, Loss 0.570, Test bal acc 0.822
Step: 40, Loss 0.615, Test bal acc 0.799
Step: 50, Loss 0.602, Test bal acc 0.801
Step: 60, Loss 0.663, Test bal acc 0.874
Step: 70, Loss 0.594, Test bal acc 0.856
Step: 80, Loss 0.654, Test bal acc 0.876
Step: 90, Loss 0.715, Test bal acc 0.825
